In [ ]:
# [셀 1] 준비: 경로, 한글 폰트, 그래프 저장 폴더, 데이터 불러오기
import os, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.font_manager as fm

BASE = Path.home() / "서울시 공영주차장 예측"
ML = BASE / "jariisseoul-repo" / "ml"
FIG = BASE / "report_figures"                      # 페이지에 넣을 그래프 저장 폴더 (저장소 밖이라 다시 받아도 안 지워짐)
FIG.mkdir(exist_ok=True)
sys.path.insert(0, str(ML))
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]
                                    if f in {x.name for x in fm.fontManager.ttflist}), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)

def save(fig, name):
    """그래프를 페이지용 PNG로 저장"""
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")

from jariml import load_config
from jariml.data import resolve_data_dir
from jariml.preprocess import build_tables

cfg = load_config()
DATA = resolve_data_dir(cfg)
tb = build_tables(DATA, cfg)
lm = tb.lot_master

print("데이터 폴더:", DATA)
print("기간:", tb.occ.index.min(), "~", tb.occ.index.max())
print("전체 주차장:", len(lm), "곳 / 예측 대상:", int(lm.use_for_model.sum()), "곳")
print("그래프 저장 폴더:", FIG)

In [ ]:
# [셀 1-1] 그래프 폰트: Pretendard (무료 폰트, PC 설치 없이 이 노트북에서만 사용)
import urllib.request
FONT_DIR = BASE / "fonts"
FONT_DIR.mkdir(exist_ok=True)
SRC = ["https://cdn.jsdelivr.net/npm/pretendard@1.3.9/dist/public/static/Pretendard-{}.otf",
       "https://unpkg.com/pretendard@1.3.9/dist/public/static/Pretendard-{}.otf"]

def get_font(w):
    p = FONT_DIR / f"Pretendard-{w}.otf"
    for u in SRC:
        if p.exists():
            break
        try:
            req = urllib.request.Request(u.format(w), headers={"User-Agent": "Mozilla/5.0"})
            p.write_bytes(urllib.request.urlopen(req, timeout=60).read())
        except Exception as e:
            print("실패:", u.format(w), e)
    return p

try:
    for w in ["Regular", "SemiBold"]:
        fm.fontManager.addfont(str(get_font(w)))
    plt.rcParams["font.family"] = "Pretendard"
    plt.rcParams["axes.titleweight"] = "semibold"
except Exception as e:
    print("Pretendard 사용 실패, 맑은 고딕 유지:", e)
print("폰트:", plt.rcParams["font.family"])

In [ ]:
# [셀 2] 1장 문제 정의: 지금 빈자리와 도착 시점 빈자리의 차이
from cycler import cycler
PURPLE, PURPLE_D = "#b39ddb", "#9575cd"                        # 연보라 (기본), 조금 진한 연보라 (선, 강조)
plt.rcParams["axes.prop_cycle"] = cycler(color=[PURPLE_D, PURPLE, "#d1c4e9", "#7e57c2"])

free = tb.cap * (1 - tb.occ)                                   # 시각 × 주차장 빈자리(대)
STEP = {"30분": 6, "1시간": 12, "2시간": 24}                    # 5분 칸 수

rows = []
for k, s in STEP.items():
    d = (free.shift(-s) - free).abs().stack()                  # 지금 → s칸 뒤 빈자리 변화(대)
    rows.append({"도착까지": k, "평균 변화(대)": d.mean(), "10대 이상 바뀐 비율(%)": (d >= 10).mean() * 100})
t1 = pd.DataFrame(rows).set_index("도착까지").round(1)
display(t1)

chg1h = (free.shift(-12) - free).abs()                         # 1시간 뒤 빈자리 변화(대)
rank = chg1h.mean().sort_values(ascending=False)
groups = {"변동 상위 3곳": rank.index[:3], "변동 하위 3곳": rank.index[-3:][::-1]}
day = [d for d in sorted(set(free.index.date)) if pd.Timestamp(d).dayofweek < 5][-2]   # 다 채워진 최근 평일
hour_chg = chg1h.groupby(free.index.hour).mean().mean(axis=1)

fig = plt.figure(figsize=(14, 10.5))
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 1.1], hspace=0.6, wspace=0.25)
for r, (label, lots) in enumerate(groups.items()):
    for i, lot in enumerate(lots):
        ax = fig.add_subplot(gs[r, i])
        f, cap = free.loc[str(day), lot], int(tb.cap[lot].iloc[-1])
        ax.plot(f.index.hour + f.index.minute / 60, f.values, color=PURPLE_D if r == 0 else PURPLE, lw=2)
        ax.set(title=f"{lm.PKLT_NM[lot]}\n총 {cap}면, 1시간 평균 변화 {rank[lot]:.1f}대",
               xlabel="시각", ylabel=f"{label}\n빈자리(대)" if i == 0 else None, xticks=range(0, 25, 6),
               ylim=(0, cap * 1.05))                           # 세로축 0 ~ 총면 고정 (작은 변화가 과장돼 보이지 않게)
        ax.title.set_fontsize(10); ax.grid(alpha=0.3)
ax = fig.add_subplot(gs[2, :])
ax.bar(hour_chg.index, hour_chg.values, color=PURPLE)
ax.set(title=f"출발 시각별 1시간 뒤 빈자리 변화 (예측 대상 {free.shape[1]}곳 전체 평균)", xlabel="출발 시각", ylabel="평균 변화(대)", xticks=range(0, 24))
ax.grid(axis="y", alpha=0.3)
fig.suptitle(f"주차장별 하루 빈자리 ({day}, 평일)", y=0.995, fontsize=12)
save(fig, "1_problem"); plt.show()
for label, lots in groups.items():
    print(f"{label}:", ", ".join(f"{lm.PKLT_NM[l]} ({rank[l]:.1f}대)" for l in lots))
print(f"1시간 뒤 빈자리가 가장 크게 바뀌는 시간대: {hour_chg.idxmax()}시 출발 (평균 {hour_chg.max():.1f}대)")

In [ ]:
# [셀 3] 2장 데이터 수집: 기간, 횟수, 수집 성공률, 데이터 종류별 규모
from jariml.data import load_live
live = load_live(DATA)
runs = pd.to_datetime(live.collected_at).dt.floor("5min").drop_duplicates().sort_values()
expected = pd.date_range(runs.min(), runs.max(), freq="5min")
gap_min = runs.diff().dt.total_seconds().div(60)

t2 = pd.Series({
    "수집 기간": f"{runs.min():%m/%d %H:%M} ~ {runs.max():%m/%d %H:%M}",
    "수집 일수": f"{(runs.max() - runs.min()).total_seconds() / 86400:.1f}일",
    "수집 주기": "주차 5분 / 도시데이터 10분",
    "주차 수집 횟수": f"{len(runs):,}회 (예정 {len(expected):,}회)",
    "주차 수집 성공률": f"{len(runs) / len(expected) * 100:.1f}%",
    "가장 긴 수집 공백": f"{gap_min.max():.0f}분",
    "주차장 수": f"{live.PKLT_CD.nunique()}곳",
    "주차 원본 행 수": f"{len(live):,}행",
}, name="값").to_frame()
display(t2)

# 데이터 종류별 파일 수, 행 수
KO = {"population": "인구", "forecast": "인구 예측", "weather": "날씨", "road": "도로 소통",
      "commercial": "상권", "events": "문화행사", "parking": "주변 주차장"}
kinds = {"공영주차장 실시간": DATA / "live", "주차장 기본 정보": DATA / "static"}
kinds |= {f"도시데이터: {KO.get(p.name, p.name)}": p for p in sorted((DATA / "citydata").glob("*")) if p.is_dir()}
count = lambda f: sum(1 for _ in open(f, encoding="utf-8", errors="ignore")) - 1
t3 = pd.DataFrame([{"데이터": k, "파일 수": len(list(p.glob("*.csv"))), "행 수": sum(count(f) for f in p.glob("*.csv"))}
                   for k, p in kinds.items()]).set_index("데이터")
display(t3)

# 하루 수집 횟수 (정상이면 하루 288회)
daily = runs.groupby(runs.dt.date).size()
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.bar([str(d)[5:] for d in daily.index], daily.values, color=PURPLE)
ax.axhline(288, color=PURPLE_D, ls="--", lw=1.5)
ax.text(len(daily) - 0.5, 290, "정상 288회 (5분 × 24시간)", ha="right", va="bottom", color=PURPLE_D)
ax.set(title="하루 주차 데이터 수집 횟수", xlabel="날짜", ylabel="수집 횟수", ylim=(0, 320))
ax.tick_params(axis="x", rotation=60); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); save(fig, "2_collection"); plt.show()

In [ ]:
# [셀 4] 3장 전처리: 예측 대상 선정, 공백 보간, 데이터 한계
p = cfg["preprocess"]
RULE = {"미연계": "실시간 연계가 안 된 주차장 (빈자리 정보 없음)",
        "버스전용": "버스 전용 주차장 (일반 승용차 이용 불가)",
        "20면 미만": f"총면 {p['min_cap']}면 미만 (1대 변화가 너무 커서 예측 의미가 작음)",
        "센서 멈춤": f"하루 점유율 변동폭 중앙값 {p['stuck_daily_range'] * 100:.0f}%p 미만 (값이 거의 안 바뀜)",
        "센서 이상": f"수집 기간 전체에서 값 종류 {p['min_unique_values']}개 미만"}
reason = lm.exclude_reason.replace("", "예측 대상")
t4 = (reason.value_counts().rename("주차장 수").to_frame()
      .assign(기준=lambda d: d.index.map(RULE).fillna("위 기준을 모두 통과")))
display(t4)

# 공백 보간: 원본에서 빈 칸 → 15분 이하만 앞뒤 값으로 채움
raw = (live.pivot_table(index="t", columns="PKLT_CD", values="NOW_PRK_VHCL_CNT", aggfunc="last")
       .reindex(index=tb.occ.index, columns=tb.occ.columns))
n_all, n_raw, n_left = raw.size, int(raw.isna().sum().sum()), int(tb.occ.isna().sum().sum())
t5 = pd.Series({"전체 칸 (시각 × 주차장)": f"{n_all:,}",
                "원본 빈 칸": f"{n_raw:,} ({n_raw / n_all * 100:.2f}%)",
                f"보간으로 채운 칸 ({p['interp_limit'] * 5}분 이하 공백)": f"{n_raw - n_left:,}",
                "그대로 비워 둔 칸 (긴 공백, 미연계 시간)": f"{n_left:,} ({n_left / n_all * 100:.2f}%)"}, name="값").to_frame()
display(t5)

# 데이터 한계: 5분 만에 총면의 30% 이상 빈자리가 바뀐 경우 (실제 이동보다 집계 초기화로 보임)
jump = (free.diff().abs() / tb.cap) >= 0.3
jump_lots = jump.sum()[lambda s: s > 0].sort_values(ascending=False)
jump_hour = jump.stack()[lambda s: s].index.get_level_values(0).hour
print(f"5분 급변(총면 30% 이상) 주차장: {len(jump_lots)}곳, 총 {int(jump_lots.sum())}회, "
      f"그중 0~5시 발생 {np.isin(jump_hour, range(0, 6)).mean() * 100:.0f}%")
print("급변 많은 곳:", ", ".join(f"{lm.PKLT_NM[l]} {n}회" for l, n in jump_lots.head(5).items()))

# 그래프: 126곳 → 예측 대상
order = ["미연계", "버스전용", "20면 미만", "센서 멈춤", "센서 이상", "예측 대상"]
cnt = reason.value_counts().reindex(order).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(10, 3.8))
bars = ax.barh(order[::-1], cnt[order[::-1]].values, color=[PURPLE_D] + [PURPLE] * 5)
ax.bar_label(bars, padding=4)
ax.set(title=f"예측 대상 선정: 전체 {len(lm)}곳 → {cnt['예측 대상']}곳", xlabel="주차장 수", xlim=(0, cnt.max() * 1.15))
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); save(fig, "3_selection"); plt.show()

In [ ]:
# [셀 5] 4.1 시간대, 요일 패턴: 언제 붐비나
from jariml.preprocess import is_offday
occ = tb.occ * 100                                              # 점유율(%)
idx = occ.index.to_series()
off = is_offday(idx, cfg["preprocess"]["holidays"]).values
hod = idx.dt.hour + idx.dt.minute / 60

wd = occ[~off].groupby(hod[~off].values).mean().mean(axis=1)    # 평일 시간대별 평균 점유율 (57곳 평균)
we = occ[off].groupby(hod[off].values).mean().mean(axis=1)      # 주말, 공휴일
DOW = "월화수목금토일"
heat = occ.mean(axis=1).groupby([idx.dt.dayofweek, idx.dt.hour]).mean().unstack()   # 요일 × 시각
hm = lambda h: f"{int(round(h * 60)) // 60}:{int(round(h * 60)) % 60:02d}"          # 13.9167 → "13:55"

t6 = pd.DataFrame({"평일": [f"{wd.max():.1f}", hm(wd.idxmax()), f"{wd.min():.1f}", hm(wd.idxmin())],
                   "주말/공휴일": [f"{we.max():.1f}", hm(we.idxmax()), f"{we.min():.1f}", hm(we.idxmin())]},
                  index=["가장 붐빌 때 점유율(%)", "가장 붐비는 시각", "가장 한산할 때 점유율(%)", "가장 한산한 시각"])
display(t6)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.1, 1]})
ax[0].plot(wd.index, wd.values, color=PURPLE_D, lw=2.2, label="평일")
ax[0].plot(we.index, we.values, color=PURPLE, lw=2.2, ls="--", label="주말/공휴일")
ax[0].set(title="시간대별 평균 점유율 (예측 대상 57곳 평균)", xlabel="시각", ylabel="점유율(%)", xticks=range(0, 25, 3))
ax[0].legend(); ax[0].grid(alpha=0.3)
im = ax[1].imshow(heat.values, aspect="auto", cmap="Purples")
ax[1].set(title="요일 × 시각 평균 점유율(%)", xlabel="시각", yticks=range(7), yticklabels=list(DOW), xticks=range(0, 24, 3))
fig.colorbar(im, ax=ax[1], fraction=0.04)
plt.tight_layout(); save(fig, "4_1_pattern"); plt.show()
print(f"평일 - 주말 점유율 차이가 가장 큰 시각: {hm((wd - we).idxmax())} ({(wd - we).max():.1f}%p)")

In [ ]:
# [셀 6] 4.2 시계열 분해 (Prophet): 기본 vs 평일/주말 하루 주기 분리
from prophet import Prophet
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

full = tb.occ.notna().all(axis=1)                               # 57곳이 모두 수집된 시각만 (빠진 곳이 있으면 전체 점유율이 왜곡됨)
y = (tb.occ * tb.cap).sum(axis=1)[full] / tb.cap.sum(axis=1)[full] * 100   # 전체 점유율(%) = 전체 주차 대수 / 전체 총면
df = pd.DataFrame({"ds": y.index, "y": y.values})
df["is_offday"] = is_offday(df.ds, cfg["preprocess"]["holidays"]).values
df["is_weekday"] = ~df.is_offday
hol = pd.DataFrame({"holiday": "공휴일", "ds": pd.to_datetime(cfg["preprocess"]["holidays"])})
CARS_PER_PP = tb.cap[full].sum(axis=1).mean() / 100             # 1%p = 몇 대

def fit(cond):
    """cond=False: 하루 주기 1개 (기본). cond=True: 평일용, 주말/공휴일용 하루 주기를 따로 (조건부 계절성)."""
    m = Prophet(daily_seasonality=not cond, weekly_seasonality=True, yearly_seasonality=False, holidays=hol)
    if cond:
        m.add_seasonality("daily_weekday", period=1, fourier_order=10, condition_name="is_weekday")
        m.add_seasonality("daily_offday", period=1, fourier_order=10, condition_name="is_offday")
    m.fit(df)
    f = m.predict(df[["ds", "is_weekday", "is_offday"]])
    f["y"], f["error"] = df.y.values, df.y.values - f.yhat.values
    return f

fc0, fc = fit(False), fit(True)                                  # fc = 조건부 (다음 셀에서도 사용)
score = lambda f: {"설명력(%)": (1 - f.error.var() / f.y.var()) * 100, "오차 평균(%p)": f.error.abs().mean(),
                   "오차 평균(대)": f.error.abs().mean() * CARS_PER_PP, "오차 표준편차(%p)": f.error.std()}
t7 = pd.DataFrame({"기본 (하루 주기 1개)": score(fc0), "평일/주말 하루 주기 분리": score(fc)}).T.round(2)
display(t7)

comp = {"추세": fc.trend, "하루 주기 (평일)": fc.daily_weekday[df.is_weekday], "하루 주기 (주말/공휴일)": fc.daily_offday[df.is_offday],
        "주간 주기": fc.weekly, "휴일": fc.holidays, "오차": fc.error}
t8 = pd.DataFrame({"최소(%p)": {k: v.min() for k, v in comp.items()}, "최대(%p)": {k: v.max() for k, v in comp.items()}})
t8["변동폭(%p)"] = t8["최대(%p)"] - t8["최소(%p)"]
t8["변동폭(대)"] = (t8["변동폭(%p)"] * CARS_PER_PP).round(0).astype(int)
display(t8.round(2))
print(f"사용한 시각: {full.sum():,}개 / 전체 {len(full):,}개, 1%p = 약 {CARS_PER_PP:.0f}대")

# 그래프 1장: 실제 vs 두 모델 (최근 7일), 평일/주말 하루 주기, 주간 주기
last = df.ds >= df.ds.max() - pd.Timedelta(days=7)
hours = lambda f, mask: f[mask].groupby(f.ds[mask].dt.hour + f.ds[mask].dt.minute / 60)
fig = plt.figure(figsize=(14, 7))
gs = fig.add_gridspec(2, 2, hspace=0.4, wspace=0.2)
a = fig.add_subplot(gs[0, :])
a.plot(df.ds[last], df.y[last], color=PURPLE, lw=2.4, label="실제")
a.plot(df.ds[last], fc0.yhat[last], color="#c5bce6", lw=1.2, ls=":", label="기본 Prophet")
a.plot(df.ds[last], fc.yhat[last], color=PURPLE_D, lw=1.3, ls="--", label="평일/주말 분리 Prophet")
a.set(title="전체 점유율: 실제 vs Prophet (최근 7일)", ylabel="점유율(%)"); a.legend(ncol=3); a.grid(alpha=0.3)
b = fig.add_subplot(gs[1, 0])
b.plot(*zip(*hours(fc, df.is_weekday).daily_weekday.mean().items()), color=PURPLE_D, lw=2, label="평일")
b.plot(*zip(*hours(fc, df.is_offday).daily_offday.mean().items()), color=PURPLE, lw=2, ls="--", label="주말/공휴일")
b.axhline(0, color="gray", lw=0.8)
b.set(title="하루 주기 (시각별 평소보다 + / -)", xlabel="시각", ylabel="%p", xticks=range(0, 25, 3)); b.legend(); b.grid(alpha=0.3)
c = fig.add_subplot(gs[1, 1])
wk = fc.groupby(fc.ds.dt.dayofweek).weekly.mean()
c.bar(list(DOW), wk.values, color=[PURPLE_D if v >= 0 else PURPLE for v in wk.values])
c.axhline(0, color="gray", lw=0.8)
c.set(title="주간 주기 (요일별 평소보다 + / -)", ylabel="%p"); c.grid(axis="y", alpha=0.3)
save(fig, "4_2_prophet"); plt.show()

In [ ]:
# [셀 7] 4.2 Fourier 분석: 어떤 주기가 얼마나 센가, Fourier order 몇이면 충분한가
from scipy.signal import find_peaks

yy = y.reindex(tb.occ.index).interpolate(limit_direction="both")      # 5분 간격을 빈틈없이 (빠진 37칸만 보간)
th = (yy.index - yy.index[0]).total_seconds().values / 3600           # 시간(h)
N = len(yy)
yd = yy.values - np.polyval(np.polyfit(th, yy.values, 1), th)        # linear detrend → 주기 성분만

# (1) FFT power spectrum: Parseval 정리로 주기별 variance share(%) 계산
X = np.fft.rfft(yd)
share = 2 * np.abs(X[1:]) ** 2 / N ** 2 / yd.var() * 100             # k번째 주파수의 variance share(%)
freq = np.arange(1, len(X)) / (N * 5 / 60)                            # 주파수 (cycles/hour)
period = 1 / freq                                                     # 주기 (hour)
df_res = freq[0]                                                      # FFT frequency resolution
D, W = 1 / 24, 1 / 168
KNOWN = {"24h (daily)": D, "12h (2nd harmonic)": 2 * D, "8h (3rd harmonic)": 3 * D, "168h (weekly)": W,
         "sideband (24h - weekly)": D - W, "sideband (24h + weekly)": D + W,
         "sideband (24h - 2·weekly)": D - 2 * W, "sideband (24h + 2·weekly)": D + 2 * W}
label = lambda f: next((k for k, v in KNOWN.items() if abs(f - v) <= df_res), f"{1 / f:.1f}h")   # resolution 안이면 같은 주기로 판정

pk, _ = find_peaks(share, height=0.2, distance=3)
peaks = (pd.DataFrame({"주기": [label(freq[k]) for k in pk], "FFT bin (h)": period[pk].round(1),
                       "variance share (%)": [share[max(k - 1, 0):k + 2].sum() for k in pk]})   # peak ±1 bin 합 (leakage 보정)
         .sort_values("variance share (%)", ascending=False).head(6).reset_index(drop=True))
peaks["누적 (%)"] = peaks["variance share (%)"].cumsum()
display(peaks.round(2))
print(f"데이터 길이 {N * 5 / 60:.1f}h → FFT bin 간격 때문에 24h는 {period[np.argmin(abs(freq - D))]:.1f}h bin, "
      f"168h는 {period[np.argmin(abs(freq - W))]:.1f}h bin에 잡힘")

# (2) Fourier order별 R²: daily harmonics(24h, 12h, 8h, ...)를 K개까지 넣고 least squares로 맞춤
off_m = is_offday(yy.index.to_series(), cfg["preprocess"]["holidays"]).values.astype(float)
def r2(K, weekly=0, split=False):
    cols = [np.ones(N), th] + ([off_m] if split else [])
    for k in range(1, K + 1):
        w = 2 * np.pi * k * th / 24
        for g in ([1 - off_m, off_m] if split else [1]):
            cols += [np.sin(w) * g, np.cos(w) * g]
    for k in range(1, weekly + 1):
        w = 2 * np.pi * k * th / 168
        cols += [np.sin(w), np.cos(w)]
    A = np.column_stack(cols)
    res = yy.values - A @ np.linalg.lstsq(A, yy.values, rcond=None)[0]
    return (1 - res.var() / yy.values.var()) * 100

Ks = range(1, 13)
curves = {"daily": [r2(k) for k in Ks],
          "daily + weekly": [r2(k, weekly=3) for k in Ks],
          "daily (평일/주말 분리) + weekly": [r2(k, weekly=3, split=True) for k in Ks]}
t9 = pd.DataFrame(curves, index=[f"K={k}" for k in Ks]).T.round(2)
display(t9[["K=1", "K=2", "K=3", "K=4", "K=6", "K=10"]])

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
ax[0].plot(period, share, color=PURPLE, lw=1)
OFFSET = {"24h (daily)": (0, 8), "12h (2nd harmonic)": (-55, 45), "sideband (24h - weekly)": (60, 55), "168h (weekly)": (0, 25)}
for _, r in peaks.head(4).iterrows():
    i = np.argmin(abs(period - r["FFT bin (h)"]))
    dx, dy = OFFSET.get(r["주기"], (0, 25))
    ax[0].annotate(f"{r['주기']}\n{r['variance share (%)']:.1f}%", (period[i], share[i]), xytext=(dx, dy), textcoords="offset points",
                   ha="center", va="bottom", fontsize=9, color=PURPLE_D,
                   arrowprops=None if (dx, dy) == (0, 8) else dict(arrowstyle="-", color=PURPLE, lw=0.8))
ax[0].set(xscale="log", xlim=(2, 400), ylim=(0, share.max() * 1.3), title="FFT power spectrum (주기별 variance share)",
          xlabel="Period (h, log)", ylabel="variance share (%)", xticks=[3, 6, 12, 24, 48, 168], xticklabels=["3", "6", "12", "24", "48", "168"])
for (name, v), st in zip(curves.items(), [":", "--", "-"]):
    ax[1].plot(list(Ks), v, marker="o", ms=4, ls=st, color=PURPLE_D, label=name)
ax[1].set(title="Fourier order별 R²", xlabel="Fourier order K (daily harmonics: 24h, 12h, 8h, ...)", ylabel="R² (%)", xticks=list(Ks))
ax[1].legend(loc="lower right")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); save(fig, "4_2_fourier"); plt.show()

In [ ]:
# [셀 8] 4.3 residual 분석: Prophet(평일/주말 분리)이 설명하지 못한 부분 = "오늘이 평소와 다른 정도"
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

res = pd.Series(fc.error.values, index=df.ds).reindex(tb.occ.index).interpolate(limit_direction="both")   # 5분 간격 빈틈없이
a = acf(res, nlags=288, fft=True)
band = 1.96 / np.sqrt(len(res))                                   # 95% 신뢰 band (이 안이면 white noise와 구분 안 됨)
half = next(k for k in range(1, 289) if a[k] < 0.5)               # half-life: ACF가 0.5 아래로 내려가는 시차
decor = next(k for k in range(1, 289) if abs(a[k]) < band)        # decorrelation: band 안으로 들어가는 시차
lb = acorr_ljungbox(res, lags=[12, 24], return_df=True)
fmt = lambda k: f"{k * 5 // 60}h {k * 5 % 60:02d}m" if k * 5 >= 60 else f"{k * 5}m"

t10 = pd.Series({"mean (%p)": res.mean(), "std (%p)": res.std(), "MAE (%p)": res.abs().mean(),
                 "MAE (대)": res.abs().mean() * CARS_PER_PP, "|residual| ≤ 1%p 비율 (%)": (res.abs() <= 1).mean() * 100,
                 "|residual| ≤ 2%p 비율 (%)": (res.abs() <= 2).mean() * 100,
                 "skewness": stats.skew(res), "excess kurtosis": stats.kurtosis(res)}, name="값").round(2).to_frame()
display(t10)

LAGS = {"5m": 1, "30m": 6, "1h": 12, "2h": 24, "4h": 48, "12h": 144, "24h": 288}
t11 = pd.DataFrame({"ACF": [a[k] for k in LAGS.values()]}, index=LAGS.keys()).T.round(3)
display(t11)
print(f"half-life: {fmt(half)} (ACF < 0.5), decorrelation: {fmt(decor)} (95% band 안)")
print(f"Ljung-Box p-value: 1h {lb.lb_pvalue.iloc[0]:.1e}, 2h {lb.lb_pvalue.iloc[1]:.1e}  → 0에 가까우면 residual은 white noise가 아님 (예측 가능한 구조가 남아 있음)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].hist(res, bins=60, density=True, color=PURPLE, alpha=0.9, label="residual")
xs = np.linspace(res.min(), res.max(), 200)
ax[0].plot(xs, stats.norm.pdf(xs, res.mean(), res.std()), color=PURPLE_D, lw=2, ls="--", label="Normal (같은 mean, std)")
ax[0].set(title="Residual 분포", xlabel="residual (%p)", ylabel="density"); ax[0].legend()
lags = np.arange(len(a))
ax[1].fill_between(lags, -band, band, color="#e8e2f6", label="95% band")
ax[1].vlines(lags, 0, a, color=PURPLE, lw=1)
ax[1].axhline(0, color="gray", lw=0.8)
for k, name, off in [(12, "1h", (35, -5)), (24, "2h", (35, -5)), (decor, "decorrelation", (15, 25))]:
    ax[1].annotate(f"{name}  {a[k]:.2f}", (k, a[k]), xytext=off, textcoords="offset points", fontsize=9, color=PURPLE_D,
                   va="center", arrowprops=dict(arrowstyle="-", color=PURPLE_D, lw=0.8))
ax[1].set(title="Residual ACF (lag 1 = 5분, 288 = 24h)", xlabel="lag", ylabel="ACF", ylim=(-0.5, 1.05), xticks=range(0, 289, 48))
ax[1].legend(loc="upper right")
for x in ax: x.grid(alpha=0.3)
plt.tight_layout(); save(fig, "4_3_residual"); plt.show()

In [ ]:
# [셀 9] 4.4 주차장별 패턴: 평일 하루 모양으로 KMeans clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

offday = is_offday(occ.index.to_series(), cfg["preprocess"]["holidays"]).values      # 주말/공휴일 여부 (다시 계산)
prof = occ[~offday].groupby(occ.index[~offday].hour).mean().T.dropna()               # 주차장 × 24시간 평일 평균 점유율(%)
Z = prof.values
sil = pd.Series({k: silhouette_score(Z, KMeans(k, n_init=20, random_state=0).fit_predict(Z)) for k in range(2, 7)})
K = int(sil.idxmax())                                                 # silhouette가 가장 높은 k
km = KMeans(K, n_init=20, random_state=0).fit(Z)
grp = pd.Series(km.labels_, index=prof.index)
cen = pd.DataFrame(km.cluster_centers_, columns=prof.columns)

def group_name(c):
    peak, amp, low, high = int(c.idxmax()), c.max() - c.min(), c.min(), c.max()
    if low >= 60: return "상시 혼잡 (밤에도 60% 이상)"
    if peak >= 19 or peak <= 6: return "주거형 (저녁, 밤 정점)"
    if high >= 70: return "업무형 혼잡 (낮 정점, 밤 여유)"
    return "업무형 여유 (낮에도 최대 50% 미만)"
order = (cen.max(axis=1) - cen.min(axis=1)).sort_values(ascending=False).index   # 진폭 큰 순서로 A, B, C ...
LETTER = {c: chr(65 + i) for i, c in enumerate(order)}
t12 = pd.DataFrame([{
    "그룹": f"{LETTER[c]}. {group_name(cen.loc[c])}", "주차장 수": int((grp == c).sum()),
    "정점 시각": f"{int(cen.loc[c].idxmax())}시", "정점 점유율(%)": cen.loc[c].max(), "최저 점유율(%)": cen.loc[c].min(),
    "하루 진폭(%p)": cen.loc[c].max() - cen.loc[c].min(), "평균 총면": tb.cap[grp.index[grp == c]].iloc[-1].mean(),
    "대표 주차장": ", ".join(lm.PKLT_NM[grp.index[grp == c]].str.replace(r"\s*공영주차장.*", "", regex=True).head(3))}
    for c in order]).set_index("그룹").round(1)
display(sil.rename("silhouette").rename_axis("k").to_frame().T.round(3))
display(t12)

STYLE = [(PURPLE_D, "-"), (PURPLE, "--"), ("#7e57c2", ":"), ("#d1c4e9", "-."), ("#5e35b1", "-"), ("#b39ddb", ":")]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1.2, 1]})
for c, (col, ls) in zip(order, STYLE):
    lab = f"{LETTER[c]}. {group_name(cen.loc[c])} ({int((grp == c).sum())}곳)"
    ax[0].plot(cen.columns, cen.loc[c], color=col, ls=ls, lw=2.2, label=lab)
    pts = prof.loc[grp.index[grp == c]]
    ax[1].scatter(pts.idxmax(axis=1), pts.max(axis=1) - pts.min(axis=1), color=col, s=40, alpha=0.8,
                  marker="os^Dv*"[list(order).index(c)], label=LETTER[c])
ax[0].set(title=f"그룹별 평일 하루 패턴 (KMeans, k={K})", xlabel="시각", ylabel="평균 점유율(%)", xticks=range(0, 24, 3))
ax[0].legend(fontsize=9)
ax[1].set(title="주차장별 정점 시각 vs 하루 진폭", xlabel="정점 시각", ylabel="하루 진폭(%p)", xticks=range(0, 24, 3))
ax[1].legend(title="그룹")
for x in ax: x.grid(alpha=0.3)
plt.tight_layout(); save(fig, "4_4_cluster"); plt.show()

In [ ]:
# [셀 10] 4.5 외부 데이터 효과 검증: 도시데이터 6종을 넣으면 1시간 뒤 예측이 좋아지나 (add-one / leave-one-out ablation)
from sklearn.ensemble import HistGradientBoostingRegressor

H = 12                                                               # 1시간 뒤 (5분 × 12)
occ_m, cap_m = tb.occ, tb.cap
amap = pd.read_csv(DATA / "reference" / "lot_area_map.csv", dtype={"PKLT_CD": str}).query("rank == 1").set_index("PKLT_CD")
LOTS = [c for c in occ_m.columns if c in amap.index]                 # 서울시 도시데이터 지역과 연결된 주차장만
end = occ_m.index.max().normalize()
split = end - pd.Timedelta(days=cfg["split"]["test_days"] - 1)       # 마지막 4일 = test (모델 평가와 같은 기준)

def city(k):
    d = pd.concat([pd.read_csv(f) for f in (DATA / "citydata" / k).glob("*.csv")])
    d["avail"] = pd.to_datetime(d.collected_at)                      # 수집 시각 = 그 정보를 알 수 있는 시각 (미래 정보 차단)
    return d
pop = city("population"); pop["ppl"] = (pop.AREA_PPLTN_MIN + pop.AREA_PPLTN_MAX) / 2
pop["congest"] = pop.AREA_CONGEST_LVL.map({"여유": 0, "보통": 1, "약간 붐빔": 2, "붐빔": 3}); pop["nonres"] = pop.NON_RESNT_PPLTN_RATE
fcp = city("forecast"); fcp["fppl"] = (fcp.FCST_PPLTN_MIN + fcp.FCST_PPLTN_MAX) / 2; fcp["target_hour"] = pd.to_datetime(fcp.FCST_TIME)
wt = city("weather"); wt["rain"] = (wt.PRECPT_TYPE != "없음").astype(float)
wt["rain_mm"] = pd.to_numeric(wt.PRECIPITATION.astype(str).str.extract(r"([\d.]+)")[0], errors="coerce").fillna(0)
wt["rain_next"] = pd.to_numeric(wt.NEXT1H_RAIN_CHANCE, errors="coerce"); wt["temp"], wt["pm10"] = wt.TEMP, pd.to_numeric(wt.PM10, errors="coerce")
road = city("road"); road["road_spd"] = road.ROAD_TRAFFIC_SPD
cm = city("commercial"); cm["pay_cnt"] = cm.AREA_SH_PAYMENT_CNT
ev = city("events"); ev[["s", "e"]] = ev.EVENT_PERIOD.str.split("~", expand=True).iloc[:, :2]
ev["s"], ev["e"] = pd.to_datetime(ev.s, errors="coerce"), pd.to_datetime(ev.e, errors="coerce")
evd = pd.DataFrame([(r.AREA_CD, d.date()) for r in ev.drop_duplicates(["AREA_CD", "EVENT_NM"]).dropna(subset=["s", "e"]).itertuples()
                    for d in pd.date_range(max(r.s, occ_m.index.min().normalize()), min(r.e, end))], columns=["AREA_CD", "date"])
evd = evd.value_counts().rename("n_event").reset_index()

def asof(X, src, cols, tol="40min"):
    src = src.dropna(subset=["avail"]).sort_values("avail")[["AREA_CD", "avail"] + cols]
    return pd.merge_asof(X.sort_values("t"), src, left_on="t", right_on="avail", by="AREA_CD",
                         direction="backward", tolerance=pd.Timedelta(tol)).drop(columns="avail")

rows = []
for c in LOTS:
    s = occ_m[c]
    d = pd.DataFrame({"t": s.index, "occ": s.values, "PKLT_CD": c, "cap": cap_m[c].values, "y": s.shift(-H).values})
    for k in (1, 3, 6, 12): d[f"d{k}"] = (s - s.shift(k)).values
    rows.append(d)
X = pd.concat(rows).dropna(subset=["occ", "y", "d12"])
X["tod"] = X.t.dt.hour * 12 + X.t.dt.minute // 5; X["ttod"] = (X.tod + H) % 288
X["dow"] = X.t.dt.dayofweek; X["off"] = is_offday(X.t, cfg["preprocess"]["holidays"]).astype(int)
X["AREA_CD"] = X.PKLT_CD.map(amap.AREA_CD)
X = asof(X, pop.rename(columns={"PPLTN_TIME": "pt"}), ["ppl", "congest", "nonres", "pt"])
X["target_hour"] = (X.t + pd.Timedelta(minutes=5 * H)).dt.round("h")
key = fcp[["AREA_CD", "PPLTN_TIME", "target_hour", "fppl"]].rename(columns={"PPLTN_TIME": "pt"}).drop_duplicates(["AREA_CD", "pt", "target_hour"])
X = X.merge(key, on=["AREA_CD", "pt", "target_hour"], how="left"); X["fppl_chg"] = X.fppl / X.ppl - 1
X = asof(X, wt, ["rain", "rain_mm", "rain_next", "temp", "pm10"]); X = asof(X, road, ["road_spd"]); X = asof(X, cm, ["pay_cnt"])
X["date"] = X.t.dt.date; X = X.merge(evd, on=["AREA_CD", "date"], how="left"); X["n_event"] = X.n_event.fillna(0)

tr, te = X[X.t < split - pd.Timedelta(minutes=5 * H)].copy(), X[X.t >= split].copy()   # 경계에서 정답이 겹치지 않게 1시간 띄움
prof_ = tr.groupby(["PKLT_CD", "off", "tod"]).occ.mean()
for d in (tr, te):
    d["p_now"] = prof_.reindex(list(zip(d.PKLT_CD, d.off, d.tod))).values
    d["p_fut"] = prof_.reindex(list(zip(d.PKLT_CD, d.off, d.ttod))).values
    d["p_delta"], d["gap"] = d.p_fut - d.p_now, d.occ - d.p_now
    for col, src in [("ppl_vs", "ppl"), ("road_vs", "road_spd")]:              # 지역의 평소 대비 (지금 / 평소)
        u = tr.groupby(["AREA_CD", "off", "tod"])[src].mean()
        d[col] = d[src] / u.reindex(list(zip(d.AREA_CD, d.off, d.tod))).values
tr, te = tr.dropna(subset=["p_now", "p_fut"]), te.dropna(subset=["p_now", "p_fut"])

BASE = ["occ", "d1", "d3", "d6", "d12", "tod", "ttod", "dow", "off", "cap", "p_now", "p_fut", "p_delta", "gap"]
G = {"인구 (population)": ["ppl", "congest", "nonres", "ppl_vs"], "인구 예측 (forecast)": ["fppl", "fppl_chg"],
     "날씨 (weather)": ["rain", "rain_mm", "rain_next", "temp", "pm10"], "도로 (road)": ["road_spd", "road_vs"],
     "상권 (commercial)": ["pay_cnt"], "문화행사 (events)": ["n_event"]}
ALL = BASE + sum(G.values(), [])
KW = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=100, random_state=0)
def mae(cols):
    m = HistGradientBoostingRegressor(**KW).fit(tr[cols], tr.y - tr.occ)
    return float((((te.occ + m.predict(te[cols])).clip(0, 1) - te.y).abs() * te.cap).mean())

base = mae(BASE)
res_ = {"주차장 데이터만": base, "주차장 + 도시데이터 6종 전부": mae(ALL)}
for k, v in G.items():
    res_[f"+ {k}"] = mae(BASE + v)
    res_[f"전부 - {k}"] = mae([c for c in ALL if c not in v])
t13 = pd.DataFrame({"1시간 뒤 MAE (대)": res_})
t13["주차장 데이터만 대비 (대)"] = t13["1시간 뒤 MAE (대)"] - base
t13["주차장 데이터만 대비 (%)"] = t13["주차장 데이터만 대비 (대)"] / base * 100
display(t13.round(3))
print(f"주차장 {len(LOTS)}곳 / train {len(tr):,}행, test {len(te):,}행 (test {split:%m/%d} ~ {end:%m/%d})")

add = t13.loc[[f"+ {k}" for k in G], "주차장 데이터만 대비 (%)"].sort_values()
fig, ax = plt.subplots(figsize=(10, 3.8))
bars = ax.barh([k[2:] for k in add.index], add.values, color=[PURPLE_D if v < 0 else PURPLE for v in add.values])
ax.bar_label(bars, labels=[f"{v:+.1f}%" for v in add.values], padding=4)
ax.axvline(0, color="gray", lw=1)
lim = max(abs(add).max() * 1.4, 1)
ax.set(xlim=(-lim, lim), title="도시데이터를 하나씩 더했을 때 1시간 뒤 MAE 변화 (주차장 데이터만 대비, - 가 개선)", xlabel="MAE 변화 (%)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); save(fig, "4_5_ablation"); plt.show()

In [ ]:
# [셀 11] 5장 모델 설계: 무엇을 예측하나, 변수 16개와 EDA 근거, 시간 순 분할
from jariml.features import FEATURES
from jariml.split import time_split

t14 = pd.DataFrame([
    ("예측 대상 (target)", "도착 시각의 점유율 변화량 = 도착 시 점유율 - 지금 점유율", "4.3 residual이 몇 시간 이어짐 → 지금 값에서 얼마나 바뀔지만 배우면 쉬움"),
    ("예측 거리", "5분 ~ 2시간 (5분 단위)", "1장: TMAP 이동 시간 대부분 2시간 안, 4.3: 2h ACF 0.71로 예측 여지 큼"),
    ("최종 출력 1", "예상 빈자리 (대) = 총면 × (1 - 예측 점유율)", "사용자가 바로 이해하는 단위"),
    ("최종 출력 2", "자리 있을 확률 = P(빈자리 ≥ 3대)", "validation residual 분포로 계산 (7.4에서 검증)"),
    ("최종 출력 3", "예측 신뢰도 (높음 / 보통 / 낮음)", "주차장별 validation 1h 오차율 2% / 4% 기준"),
], columns=["항목", "정의", "근거"]).set_index("항목")
display(t14)

WHY = {
    "h": ("예측 거리", "1장: 도착까지 멀수록 변화가 큼 (30m 6.4대 → 2h 22.6대)"),
    "occ": ("지금 점유율", "4.3: residual ACF 1h 0.85 → 지금 상태가 이어짐"),
    "d1": ("최근 5분 변화", "단기 추세 (차가 들어오는 중인지 빠지는 중인지)"),
    "d3": ("최근 15분 변화", "단기 추세"), "d6": ("최근 30분 변화", "단기 추세"), "d12": ("최근 60분 변화", "단기 추세"),
    "tod": ("지금 시각", "4.1, 4.2: 24h 주기가 분산의 66.8%"),
    "ttod": ("도착 시각", "4.2: 도착 시각의 하루 주기 위치"),
    "dow": ("요일", "4.2: weekly 주기 (화~목 +3~4%p, 일 -7%p)"),
    "off": ("지금 휴일 여부", "4.2: 평일/주말 분리로 R² 85% → 99%"),
    "toff": ("도착 휴일 여부", "4.2: 도착 시점이 다른 날로 넘어가는 경우"),
    "cap": ("총면", "4.4: 규모와 유형에 따라 패턴이 다름"),
    "p_now": ("평소 점유율 (지금)", "4.4: 주차장마다 하루 모양이 다름 (4개 유형)"),
    "p_fut": ("평소 점유율 (도착)", "4.4: 주차장별, 평일/주말별 평소 패턴"),
    "p_delta": ("평소 변화량", "기준선 '지금 + 평소 변화'를 모델 안에 포함"),
    "gap": ("오늘 - 평소 차이", "4.3: 오늘의 이상 상태 (half-life 3h 20m)"),
}
t15 = pd.DataFrame([(f, *WHY.get(f, ("", ""))) for f in FEATURES], columns=["변수", "의미", "근거 (EDA)"]).set_index("변수")
display(t15)
print("외부 도시데이터 6종: 4.5 ablation에서 효과 없음 (전부 넣으면 +4.7%, 날씨 +5.3%) → 사용 안 함")

s = cfg["split"]
sp = time_split(tb.occ.index, s["test_days"], s["val_days"], s["gap_steps"], cfg["preprocess"]["step_min"])
seg = [("train", sp.train.min(), sp.train.max()), ("validation", sp.val.min(), sp.val.max()), ("test", sp.test.min(), sp.test.max())]
t16 = pd.DataFrame([{"구간": n, "시작": a, "끝": b, "일수": round((b - a).total_seconds() / 86400, 1),
                     "용도": {"train": "모델 학습", "validation": "조기 종료, 튜닝, 확률 분포", "test": "최종 성능 평가 (한 번만)"}[n]}
                    for n, a, b in seg]).set_index("구간")
display(t16)
print(f"구간 사이 gap: {s['gap_steps'] * 5}분 (앞 구간의 정답이 뒤 구간 입력과 겹치지 않게)")

fig, ax = plt.subplots(figsize=(13, 1.8))
for (n, a, b), col in zip(seg, ["#d1c4e9", PURPLE, PURPLE_D]):
    ax.barh(0, b - a, left=a, color=col, height=0.5)
    ax.text(a + (b - a) / 2, 0, f"{n}\n{a:%m/%d} ~ {b:%m/%d}", ha="center", va="center", fontsize=10,
            color="white" if n != "train" else "#4a3b7a")
ax.set(yticks=[], title="시간 순 분할 (미래 데이터로 과거를 맞히지 않게)", xlim=(tb.occ.index.min(), tb.occ.index.max()))
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); save(fig, "5_split"); plt.show()

In [ ]:
# [셀 12] 6장 모델 비교: 9개 (ML 5, DL 2, 기준선 2)를 같은 기간, 같은 지표로 비교 (11/17 실행 결과)
cmp = pd.read_csv(ML / "reports" / "model_comparison_20261117.csv", index_col=0)
TYPE = {"ridge": "ML (선형)", "rf": "ML (트리)", "gbm": "ML (트리, boosting)", "lgbm": "ML (트리, boosting)",
        "xgb": "ML (트리, boosting)", "mlp": "DL (MLP)", "gru": "DL (GRU, 시계열)"}
cmp.insert(0, "종류", [TYPE.get(i, "기준선") for i in cmp.index])
best = cmp["MAE 1시간(대)"].min()
cmp["1위 대비 (%)"] = (cmp["MAE 1시간(대)"] / best - 1) * 100
cols = ["종류", "MAE 30분(대)", "MAE 1시간(대)", "MAE 2시간(대)", "1위 대비 (%)", "RMSE 1시간(대)", "MSLE 1시간", "Brier",
        "예측 응답(ms)", "학습(초)", "모델 크기(MB)"]
display(cmp.sort_values("MAE 1시간(대)")[cols].round(3))
print("평가 기간: 11/14 ~ 11/17 (11/17 실행), 예측 응답 = 57곳 한 번에 예측하는 시간")

m_ = cmp.dropna(subset=["학습(초)"]).sort_values("MAE 1시간(대)")
base_ = cmp[cmp["종류"] == "기준선"]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"width_ratios": [1, 1.1]})
order_ = cmp.sort_values("MAE 1시간(대)", ascending=False)
ax[0].barh(order_.index, order_["MAE 1시간(대)"],
           color=[PURPLE_D if i == "lgbm" else ("#e8e2f6" if t == "기준선" else PURPLE) for i, t in zip(order_.index, order_["종류"])])

for i, v in enumerate(order_["MAE 1시간(대)"]):
    ax[0].text(v + 0.1, i, f"{v:.2f}", va="center", fontsize=9)
ax[0].set(title="1시간 뒤 MAE (대, 작을수록 좋음)", xlabel="MAE (대)", xlim=(0, order_["MAE 1시간(대)"].max() * 1.12))
ax[1].scatter(m_["학습(초)"], m_["MAE 1시간(대)"], s=m_["모델 크기(MB)"].clip(0.2) * 25 + 40,
              color=[PURPLE_D if i == "lgbm" else PURPLE for i in m_.index], alpha=0.9)

POS = {"lgbm": (-10, -14, "right"), "xgb": (8, 8, "left"), "gru": (-10, 6, "right")}     # 겹치거나 잘리는 이름만 위치 지정
for i, r in m_.iterrows():
    dx, dy, ha = POS.get(i, (8, 4, "left"))
    ax[1].annotate(i, (r["학습(초)"], r["MAE 1시간(대)"]), xytext=(dx, dy), textcoords="offset points", fontsize=10, ha=ha,
                   color=PURPLE_D, fontweight="semibold" if i == "lgbm" else "normal")
ax[1].set_xlim(m_["학습(초)"].min() * 0.7, m_["학습(초)"].max() * 1.6)
ax[1].axhline(base_["MAE 1시간(대)"].min(), color="gray", ls="--", lw=1)
ax[1].text(m_["학습(초)"].max(), base_["MAE 1시간(대)"].min(), "기준선 (지금 + 평소 변화)", ha="right", va="bottom", fontsize=9, color="gray")
ax[1].set(xscale="log", title="정확도 vs 학습 시간 (왼쪽 아래가 좋음, 원 크기 = 모델 크기)", xlabel="학습 시간 (초, log)", ylabel="1시간 뒤 MAE (대)")
for x in ax: x.grid(alpha=0.3)
plt.tight_layout(); save(fig, "6_compare"); plt.show()
ax[1].set_ylim(m_["MAE 1시간(대)"].min() - 0.08, m_["MAE 1시간(대)"].max() + 0.05)

In [ ]:
# [셀 13] 7.1 하이퍼파라미터 튜닝 (Optuna), 7.2 walk-forward 교차검증: 04 노트북 결과 (11/18 실행)
DEFAULT = {"objective": "l2", "num_leaves": 63, "learning_rate": 0.05, "min_child_samples": 100,
           "subsample": 0.8, "colsample_bytree": 0.9, "reg_lambda": 1.0}
tuned_p = cfg["engine_params"]["lgbm"]
DESC = {"objective": "loss function", "num_leaves": "트리 하나의 복잡도", "learning_rate": "한 번에 배우는 양",
        "min_child_samples": "leaf 하나의 최소 표본 (overfitting 방지)", "subsample": "트리마다 쓰는 행 비율",
        "colsample_bytree": "트리마다 쓰는 변수 비율", "reg_lambda": "L2 regularization"}
t17 = pd.DataFrame({"기본값": DEFAULT, "튜닝 결과": {k: tuned_p.get(k) for k in DEFAULT}, "의미": DESC})
display(t17)

t18 = pd.DataFrame({"기본값": [3.818, 4.557], "튜닝": [3.619, 4.330]}, index=["validation 1h MAE (대)", "test 1h MAE (대)"])
t18["개선 (%)"] = (t18["튜닝"] / t18["기본값"] - 1) * 100
display(t18.round(3))
print("Optuna TPE 52회 (30분 제한), 목표 = validation 1시간 뒤 MAE. test는 튜닝이 끝난 뒤 한 번만 평가")
print("loss function도 후보(l2, l1, huber)로 넣었고 l1(MAE)이 선택됨 → 4.3의 두꺼운 꼬리(excess kurtosis 1.80)와 일치")

# walk-forward: test 기간을 2일씩 과거로 옮기며 4번 평가 (각 회차는 그 시점까지의 데이터만 사용)
wf = pd.DataFrame({"기준선 (지금 + 평소 변화)": [4.164, 5.911, 4.561, 3.665],
                   "lgbm 기본값": [3.928, 5.084, 4.171, 3.489],
                   "lgbm 튜닝": [3.792, 4.836, 3.983, 3.367]},
                  index=["11/16~11/17", "11/14~11/15", "11/12~11/13", "11/10~11/11"])
wf_s = pd.DataFrame({"mean": wf.mean(), "std": wf.std(), "min": wf.min(), "max": wf.max()})
display(wf.round(3)); display(wf_s.round(3))
print(f"튜닝이 기본값보다 좋았던 회차: {(wf['lgbm 튜닝'] < wf['lgbm 기본값']).sum()} / {len(wf)}회, "
      f"평균 개선 {(wf['lgbm 튜닝'] / wf['lgbm 기본값'] - 1).mean() * 100:.1f}%")

fig, ax = plt.subplots(figsize=(12, 4))
xs, w = np.arange(len(wf)), 0.26
for j, (c, col) in enumerate(zip(wf.columns, ["#e8e2f6", PURPLE, PURPLE_D])):
    b = ax.bar(xs + (j - 1) * w, wf[c], w, color=col, label=c)
    ax.bar_label(b, fmt="%.2f", fontsize=8, padding=2)
ax.set(xticks=xs, xticklabels=[f"fold {i + 1}\n{d}" for i, d in enumerate(wf.index)], ylabel="1시간 뒤 MAE (대)",
       title="Walk-forward 교차검증 (4개 기간, 낮을수록 좋음)", ylim=(0, wf.values.max() * 1.15))
ax.legend(ncol=3, loc="upper right"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); save(fig, "7_tuning_cv"); plt.show()

In [ ]:
# [셀 14] 7.3 예측 근거 (SHAP): 튜닝된 lgbm을 최신 분할로 학습하고, 변수별 기여도 계산
import time
from jariml.engines import ENGINES
from jariml import tuning
from matplotlib.patches import Patch

t0 = time.time()
eng = ENGINES["lgbm"](cfg)                                       # config.yaml의 튜닝 설정 (l1, num_leaves 254 ...)
eng.fit(tb.occ, tb.cap, sp)                                      # train으로 학습, validation으로 early stopping
print(f"학습 {time.time() - t0:.0f}초, 트리 {eng.info['best_iter']}개 (early stopping)")

imp, S, XS = tuning.shap_table(eng, tb, sp, cfg, n=3000)          # test 1시간 뒤 예측 3,000건, TreeSHAP (LightGBM 내장)
GROUP = {"평소 패턴": ["평소 변화", "평소(도착)", "평소(지금)", "지금 시각", "도착 시각", "요일", "지금 휴일", "도착 휴일"],
         "오늘의 상태": ["지금 점유율", "오늘-평소 차이", "5분 변화", "15분 변화", "30분 변화", "60분 변화"],
         "기타": ["예측 거리", "총면"]}
t19 = imp.to_frame("mean |SHAP| (대)")
t19["비중 (%)"] = t19["mean |SHAP| (대)"] / t19["mean |SHAP| (대)"].sum() * 100
t19["그룹"] = [next(g for g, v in GROUP.items() if f in v) for f in t19.index]
display(t19.round(3))
display(t19.groupby("그룹")[["mean |SHAP| (대)", "비중 (%)"]].sum().sort_values("비중 (%)", ascending=False).round(1))

fig, ax = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1, 1.1]})
s_ = t19.sort_values("mean |SHAP| (대)")
ax[0].barh(s_.index, s_["mean |SHAP| (대)"], color=[PURPLE_D if g == "오늘의 상태" else (PURPLE if g == "평소 패턴" else "#d1c4e9") for g in s_["그룹"]])
ax[0].set(title="변수별 평균 영향 (1시간 뒤 빈자리, 대)", xlabel="mean |SHAP| (대)")
ax[0].legend(handles=[Patch(color=PURPLE, label="평소 패턴"), Patch(color=PURPLE_D, label="오늘의 상태"), Patch(color="#d1c4e9", label="기타")], loc="lower right")
ax[1].scatter(XS["오늘-평소 차이"] * 100, S["오늘-평소 차이"], s=6, alpha=0.35, color=PURPLE_D)
ax[1].axhline(0, color="gray", lw=0.8); ax[1].axvline(0, color="gray", lw=0.8)
ax[1].set(title="오늘 - 평소 차이가 예측에 주는 영향", xlabel="오늘 - 평소 차이 (%p, + = 평소보다 더 참)", ylabel="빈자리 영향 (대, + = 늘어남)")
for x in ax: x.grid(alpha=0.3)
out = int(((S["오늘-평소 차이"].abs() > 20) | ((XS["오늘-평소 차이"] * 100).abs() > 40)).sum())
ax[1].set(xlim=(-40, 40), ylim=(-20, 20), title=f"오늘 - 평소 차이가 예측에 주는 영향 (범위 밖 {out}건 생략)")
plt.tight_layout(); save(fig, "7_3_shap"); plt.show()

In [ ]:
# [셀 15] 7.4 확률 신뢰도: "자리 있을 확률 p%"라고 한 경우 실제로 p% 자리가 있었나 (reliability diagram)
prob, cal = tuning.calibration(eng, tb, sp, cfg)                 # validation residual로 확률 생성 → test에서 채점
display(pd.Series(prob, name="값").to_frame().round(4))
display(cal.round(3))
gap_ = (cal["실제비율"] - cal["예측확률평균"]).abs()
print(f"Brier score {prob['Brier']:.4f} (평균 확률만 쓸 때 {prob['Brier(항상 평균 확률)']:.4f}, "
      f"{(1 - prob['Brier'] / prob['Brier(항상 평균 확률)']) * 100:.0f}% 개선)")
print(f"구간별 |예측 - 실제| 가중 평균: {(gap_ * cal['건수']).sum() / cal['건수'].sum() * 100:.2f}%p (ECE)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.2]})
ax[0].plot([0, 1], [0, 1], ls="--", color="gray", lw=1, label="완벽한 calibration")
ax[0].plot(cal["예측확률평균"], cal["실제비율"], marker="o", color=PURPLE_D, lw=2, label="우리 모델")
ax[0].set(title="Reliability diagram", xlabel="예측한 확률", ylabel="실제 자리 있던 비율", xlim=(0, 1), ylim=(0, 1.02))
ax[0].legend(loc="upper left")
ax[1].bar(range(len(cal)), cal["건수"], color=PURPLE)
ax[1].set(yscale="log", title="확률 구간별 건수 (log)", xticks=range(len(cal)), ylabel="건수",
          xticklabels=[f"{int(lo * 100)}~{int(hi * 100)}%" for lo, hi in zip(np.arange(0, 1, 0.1), np.arange(0.1, 1.01, 0.1))][:len(cal)])
ax[1].tick_params(axis="x", rotation=45)
for x in ax: x.grid(alpha=0.3)
plt.tight_layout(); save(fig, "7_4_calibration"); plt.show()

In [ ]:
# [셀 16] 8장 최종 성능: 튜닝된 lgbm, 최신 test 기간, 예측 거리 5분 ~ 2시간 전부 채점
from jariml.evaluate import truth_frame, eval_frame
from jariml.features import fit_profile

HS = list(range(1, cfg["target"]["max_horizon_steps"] + 1))     # 5분 ~ 2시간 (24칸)
hol_ = cfg["preprocess"]["holidays"]
prof8 = fit_profile(tb.occ[tb.occ.index < sp.val_start], hol_)
E = eval_frame(truth_frame(tb.occ, tb.cap, sp.test, HS, prof8, hol_), eng.predict(tb.occ, tb.cap, sp.test, HS))
E["모델"] = (E.pred - E.y).abs() * E.cap                               # 빈자리 오차 (대)
E["지금값 유지"] = (E.occ - E.y).abs() * E.cap
E["지금 + 평소 변화"] = ((E.occ + E.p_delta).clip(0, 1) - E.y).abs() * E.cap
E["오차율"] = (E.pred - E.y).abs() * 100                              # 총면 대비 오차 (%)
E["rmse_sq"] = ((E.pred - E.y) * E.cap) ** 2
M = ["모델", "지금 + 평소 변화", "지금값 유지"]

rows8 = []
for name, h in [("30분", 6), ("1시간", 12), ("2시간", 24)]:
    g = E[E.h == h]
    rows8.append({"도착까지": name, "MAE (대)": g["모델"].mean(), "오차율 (총면 대비, %)": g["오차율"].mean(),
                  "RMSE (대)": np.sqrt(g.rmse_sq.mean()), "지금값 유지 (대)": g["지금값 유지"].mean(),
                  "지금 + 평소 변화 (대)": g["지금 + 평소 변화"].mean(),
                  "지금값 유지 대비 (%)": (g["모델"].mean() / g["지금값 유지"].mean() - 1) * 100,
                  "지금 + 평소 변화 대비 (%)": (g["모델"].mean() / g["지금 + 평소 변화"].mean() - 1) * 100})
t20 = pd.DataFrame(rows8).set_index("도착까지").round(2)
display(t20)

g1 = E[E.h == 12]
lot_rate = g1.groupby("PKLT_CD")["오차율"].mean()
t21 = pd.Series({"주차장 수": len(lot_rate), "평균 (%)": lot_rate.mean(), "중앙값 (%)": lot_rate.median(),
                 "2% 이하 (곳)": (lot_rate <= 2).sum(), "2~4% (곳)": ((lot_rate > 2) & (lot_rate <= 4)).sum(),
                 "4% 초과 (곳)": (lot_rate > 4).sum(), "최소 (%)": lot_rate.min(), "최대 (%)": lot_rate.max()},
                name="1시간 뒤 주차장별 오차율").round(2).to_frame()
display(t21)
print(f"test 기간 {sp.test.min():%m/%d %H:%M} ~ {sp.test.max():%m/%d %H:%M}, 채점 {len(E):,}건 (주차장 × 기준 시각 × 예측 거리)")

hour_ = g1.groupby(g1.t.dt.hour)[M].mean()
by_h = E.groupby("h")[M].mean()
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5), gridspec_kw={"width_ratios": [1, 1.2, 0.9]})
for c, col, ls in zip(M, [PURPLE_D, PURPLE, "#d1c4e9"], ["-", "--", ":"]):
    ax[0].plot(by_h.index * 5, by_h[c], color=col, ls=ls, lw=2.2, label=c)
    ax[1].plot(hour_.index, hour_[c], color=col, ls=ls, lw=2.2, marker="o", ms=3, label=c)
ax[0].set(title="예측 거리별 MAE", xlabel="도착까지 (분)", ylabel="MAE (대)", xticks=[5, 30, 60, 90, 120])
ax[1].set(title="출발 시각별 1시간 뒤 MAE", xlabel="출발 시각", ylabel="MAE (대)", xticks=range(0, 24, 3))
ax[0].legend(); ax[1].legend()
ax[2].hist(lot_rate, bins=20, color=PURPLE)
ax[2].axvline(lot_rate.median(), color=PURPLE_D, ls="--", lw=1.5)
ax[2].text(lot_rate.median() + 0.25, ax[2].get_ylim()[1] * 0.9, f"중앙값 {lot_rate.median():.2f}%", color=PURPLE_D, fontsize=9,
           bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.5))

ax[2].set(title=f"주차장별 1시간 뒤 오차율 ({len(lot_rate)}곳)", xlabel="오차율 (총면 대비, %)", ylabel="주차장 수")
for x in ax: x.grid(alpha=0.3)
plt.tight_layout(); save(fig, "8_final"); plt.show()